# Historical analysis notebook
Code-only copy. Outputs and original narrative cells were removed. See docs/reproduction.md for the revised analysis and execution order.


In [ ]:
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import load
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
def locate_project_root() -> Path:
    start = Path.cwd().resolve()
    for root in [start, *start.parents]:
        if (root / 'data').exists() and (root / 'results').exists():
            return root
    raise FileNotFoundError('Unable to locate project root from current working directory')

PROJECT_ROOT = locate_project_root()
FOLLOWUP_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed_follow_up'
RESULTS_DIR = PROJECT_ROOT / 'results'
IMAGES_DIR = PROJECT_ROOT / 'images'
ARTIFACTS_DIR = PROJECT_ROOT / 'models' / 'nested_loso_artifacts'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = 'VO2peak_abs'
TIME_WINDOW = (300.0, 1000.0)
WR_LEVELS = (30, 80)
MIN_SAMPLES_PER_LEVEL = 20

TIME_COLUMN_CANDIDATES = ('Time', 'time', 't')
WORKRATE_COLUMN_CANDIDATES = ('WorkRate', 'WR', 'work_rate', 'WR_cortex')
EXCLUDED_COLUMNS = {
    'Time', 'time', 't', 'Timecode',
    'WorkRate', 'workrate', 'WR', 'wr', 'work_rate', 'WR_cortex'
}

GAS_EXCHANGE_VENTILATION = [
    'VO2pulm', 'VO2_movMean', 'VCO2', 'FO2et', 'FCO2et', 'Vent', 'VT', 'BF'
]
HR_ONLY = ['HR']
HEMODYNAMICS = [
    'SV', 'CO', 'dPdt', 'SPTI', 'RPP', 'DPTI', 'DPTISPTI', 'LVET', 'Cwk',
    'Rp', 'TPR', 'fiSYS', 'fiMAP', 'fiDIA', 'sBP', 'mBP', 'dBP'
]
NIRS = ['absO2Hb_Rx1', 'absHHb_Rx1', 'TSI_Rx1_Tx1Tx2Tx3', 'TSI_FF_Rx1_Tx1Tx2Tx3']
ALL_SIGNALS = sorted({*HR_ONLY, *GAS_EXCHANGE_VENTILATION, *HEMODYNAMICS, *NIRS})
NO_GAS_EXCHANGE = sorted(set(ALL_SIGNALS) - set(GAS_EXCHANGE_VENTILATION))
HEMO_PLUS_NIRS = sorted(set(HEMODYNAMICS + NIRS))

SENSOR_GROUPS: Dict[str, Sequence[str]] = {
    'all_signals': ALL_SIGNALS,
    'no_gas_exchange': NO_GAS_EXCHANGE,
    'hemo_plus_nirs': HEMO_PLUS_NIRS,
    'nirs': NIRS,
}
GROUPS_TO_EVALUATE = ('all_signals', 'no_gas_exchange', 'hemo_plus_nirs', 'nirs')
MODELS = ('Ridge', 'ElasticNet', 'PLS')

PROJECT_ROOT, FOLLOWUP_PROCESSED_DIR, ARTIFACTS_DIR

In [ ]:
class FoldwiseUnitRescaler(BaseEstimator, TransformerMixin):
    """Compatibility class needed for loading serialized nested-LOSO pipelines."""

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X).astype(float)
        scales = X_df.std(axis=0, ddof=0).replace(0.0, 1.0).fillna(1.0)
        self.scales_ = scales.to_numpy(dtype=float)
        return self

    def transform(self, X):
        X_arr = np.asarray(X, dtype=float)
        return X_arr / self.scales_

def safe_name(value: str) -> str:
    return value.replace(' ', '_').replace('/', '_')

def detect_column(columns: Iterable[str], candidates: Sequence[str]) -> str:
    lower_map = {col.lower(): col for col in columns}
    for candidate in candidates:
        key = candidate.lower()
        if key in lower_map:
            return lower_map[key]
    raise KeyError(f'Candidate columns {candidates} not present')

def list_subject_ids(processed_dir: Path) -> List[str]:
    subjects: List[str] = []
    for data_path in sorted(processed_dir.glob('CT_*_data.csv')):
        subject_id = data_path.stem.rsplit('_', 1)[0]
        target_path = processed_dir / f'{subject_id}_targets.csv'
        if target_path.exists():
            subjects.append(subject_id)
    if not subjects:
        raise FileNotFoundError('No subject data-target pairs found')
    return subjects

def load_subject_followup(subject_id: str) -> Tuple[pd.DataFrame, float]:
    data_path = FOLLOWUP_PROCESSED_DIR / f'{subject_id}_data.csv'
    target_path = FOLLOWUP_PROCESSED_DIR / f'{subject_id}_targets.csv'
    if not data_path.exists() or not target_path.exists():
        raise FileNotFoundError(f'Missing follow-up files for {subject_id}')
    data_df = pd.read_csv(data_path)
    target_df = pd.read_csv(target_path)
    if TARGET_COLUMN not in target_df.columns:
        raise KeyError(f'Missing target {TARGET_COLUMN} in {target_path.name}')
    return data_df, float(target_df[TARGET_COLUMN].iloc[0])

def slice_prbs_window(df: pd.DataFrame, time_col: str, window: Tuple[float, float]) -> pd.DataFrame:
    start, stop = window
    out = df[(df[time_col] >= start) & (df[time_col] <= stop)].copy()
    if out.empty:
        raise ValueError(f'No data in PRBS window {window}')
    return out

def select_numeric_signals(df: pd.DataFrame, candidates: Sequence[str]) -> List[str]:
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    return [col for col in candidates if col in numeric_cols]

def extract_wr_conditioned_features(
    df: pd.DataFrame,
    signal_columns: Sequence[str],
    wr_column: str,
    min_samples: int,
) -> Dict[str, float]:
    features: Dict[str, float] = {}
    lower_wr, upper_wr = sorted(WR_LEVELS)
    for signal in signal_columns:
        mean_lower = np.nan
        mean_upper = np.nan
        for wr in WR_LEVELS:
            wr_values = df.loc[df[wr_column] == wr, signal].dropna()
            mean_name = f'{signal}_mean_wr{wr}'
            std_name = f'{signal}_std_wr{wr}'
            features[mean_name] = np.nan
            features[std_name] = np.nan
            if wr_values.shape[0] >= min_samples:
                features[mean_name] = float(wr_values.mean())
                features[std_name] = float(wr_values.std(ddof=1))
                if wr == lower_wr:
                    mean_lower = features[mean_name]
                else:
                    mean_upper = features[mean_name]

        delta_name = f'{signal}_mean_delta_wr{upper_wr}_minus_wr{lower_wr}'
        features[delta_name] = (
            mean_upper - mean_lower
            if not np.isnan(mean_lower) and not np.isnan(mean_upper)
            else np.nan
        )
    return features

def build_followup_feature_table(subject_ids: Sequence[str], group_columns: Sequence[str]) -> Tuple[pd.DataFrame, pd.Series]:
    rows: List[Dict[str, float]] = []
    targets: Dict[str, float] = {}

    for subject_id in subject_ids:
        data_df, target_value = load_subject_followup(subject_id)
        time_col = detect_column(data_df.columns, TIME_COLUMN_CANDIDATES)
        wr_col = detect_column(data_df.columns, WORKRATE_COLUMN_CANDIDATES)
        prbs_df = slice_prbs_window(data_df, time_col, TIME_WINDOW)

        if group_columns == ALL_SIGNALS:
            candidates = [col for col in prbs_df.columns if col not in EXCLUDED_COLUMNS]
            signal_columns = select_numeric_signals(prbs_df, candidates)
        else:
            signal_columns = select_numeric_signals(prbs_df, group_columns)

        if not signal_columns:
            print(f'Warning: no valid signals for {subject_id}; keeping subject with NaN feature row')
            feats = {}
        else:
            feats = extract_wr_conditioned_features(prbs_df, signal_columns, wr_col, MIN_SAMPLES_PER_LEVEL)

        feats['subject_id'] = subject_id
        rows.append(feats)
        targets[subject_id] = target_value

    X = pd.DataFrame(rows).set_index('subject_id').sort_index()
    y = pd.Series(targets, name=TARGET_COLUMN).sort_index()
    return X, y

In [ ]:
followup_subject_ids = list_subject_ids(FOLLOWUP_PROCESSED_DIR)
print('Follow-up subjects:', len(followup_subject_ids))
print(', '.join(followup_subject_ids))

followup_features_by_group: Dict[str, pd.DataFrame] = {}
followup_target_reference = None

for group_name in GROUPS_TO_EVALUATE:
    X_g, y_g = build_followup_feature_table(followup_subject_ids, SENSOR_GROUPS[group_name])
    followup_features_by_group[group_name] = X_g
    if followup_target_reference is None:
        followup_target_reference = y_g
    print(f'{group_name}: X shape={X_g.shape}, y n={len(y_g)}')

followup_target_reference.head()

In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    residuals = y_pred - y_true
    mse = mean_squared_error(y_true, y_pred)
    return {
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'RMSE': float(np.sqrt(mse)),
        'R2': float(r2_score(y_true, y_pred)),
        'MeanResidual': float(residuals.mean()),
        'StdResidual': float(residuals.std(ddof=1)),
    }

def plot_pred_vs_actual(y_true: pd.Series, y_pred: pd.Series, title: str, save_path: Path) -> None:
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, color='navy', alpha=0.8)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    plt.plot(lims, lims, '--', color='gray', label='Ideal 1:1')
    plt.xlabel('Actual VO2peak_abs (L/min)')
    plt.ylabel('Predicted VO2peak_abs (L/min)')
    plt.title(title)
    plt.xlim(lims)
    plt.ylim(lims)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

def plot_bland_altman(y_true: pd.Series, y_pred: pd.Series, title: str, save_path: Path) -> None:
    mean_values = (y_true + y_pred) / 2.0
    residuals = y_pred - y_true
    mean_residual = residuals.mean()
    std_residual = residuals.std(ddof=1)
    loa = 1.96 * std_residual

    plt.figure(figsize=(6, 4))
    plt.scatter(mean_values, residuals, color='darkorange', alpha=0.8)
    plt.axhline(mean_residual, color='black', linestyle='--', linewidth=1, label='Mean bias')
    plt.axhline(mean_residual + loa, color='red', linestyle='--', linewidth=0.8, label='Upper LOA')
    plt.axhline(mean_residual - loa, color='red', linestyle='--', linewidth=0.8, label='Lower LOA')
    plt.xlabel('Mean VO2peak_abs (L/min)')
    plt.ylabel('Residual (Predicted - Actual)')
    plt.title(title)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

def evaluate_zero_shot_combo(group_name: str, model_name: str) -> Tuple[pd.DataFrame, Dict[str, float]]:
    file_token = f"nested_{safe_name(group_name)}_{safe_name(model_name)}"
    manifest_path = RESULTS_DIR / f'{file_token}_artifacts_manifest.csv'
    if not manifest_path.exists():
        raise FileNotFoundError(f'Artifact manifest not found: {manifest_path}')

    manifest = pd.read_csv(manifest_path).set_index('subject_id')
    common_subjects = [sid for sid in followup_subject_ids if sid in manifest.index]
    if not common_subjects:
        raise ValueError(f'No overlapping subjects found in manifest for {group_name} × {model_name}')

    X_follow = followup_features_by_group[group_name]
    y_follow = followup_target_reference

    records: List[Dict[str, float]] = []
    for subject_id in common_subjects:
        artifact_path = Path(manifest.loc[subject_id, 'artifact_path'])
        if not artifact_path.exists():
            continue

        payload = load(artifact_path)
        estimator = payload['estimator']
        feature_columns = payload['feature_columns']

        x_row = X_follow.loc[[subject_id]].reindex(columns=feature_columns)
        pred = float(estimator.predict(x_row)[0])
        actual = float(y_follow.loc[subject_id])

        records.append({
            'subject_id': subject_id,
            'actual_vo2peak': actual,
            'predicted_vo2peak': pred,
            'residual': pred - actual,
            'n_missing_features': int(x_row.isna().sum(axis=1).iloc[0]),
            'n_features': int(x_row.shape[1]),
        })

    pred_df = pd.DataFrame(records).set_index('subject_id').sort_index()
    if pred_df.empty:
        raise ValueError(f'No predictions generated for {group_name} × {model_name}')

    y_true = pred_df['actual_vo2peak'].to_numpy(dtype=float)
    y_pred = pred_df['predicted_vo2peak'].to_numpy(dtype=float)
    metrics = compute_metrics(y_true, y_pred)

    if len(pred_df) >= 2:
        pearson_r, pearson_p = stats.pearsonr(y_pred, y_true)
        spearman_rho, spearman_p = stats.spearmanr(y_pred, y_true)
        calib = stats.linregress(y_pred, y_true)
    else:
        pearson_r, pearson_p = np.nan, np.nan
        spearman_rho, spearman_p = np.nan, np.nan
        calib = None

    summary = {
        'Group': group_name,
        'Model': model_name,
        'NSubjects': int(len(pred_df)),
        'MAE': metrics['MAE'],
        'RMSE': metrics['RMSE'],
        'R2': metrics['R2'],
        'MeanResidual': metrics['MeanResidual'],
        'StdResidual': metrics['StdResidual'],
        'Pearson_r': float(pearson_r),
        'Pearson_p': float(pearson_p),
        'Spearman_rho': float(spearman_rho),
        'Spearman_p': float(spearman_p),
        'Calibration_intercept': float(calib.intercept) if calib is not None else np.nan,
        'Calibration_slope': float(calib.slope) if calib is not None else np.nan,
        'Calibration_rvalue': float(calib.rvalue) if calib is not None else np.nan,
    }
    return pred_df, summary

In [ ]:
summary_records: List[Dict[str, float]] = []
error_records: List[Dict[str, str]] = []

for group_name in GROUPS_TO_EVALUATE:
    for model_name in MODELS:
        try:
            pred_df, summary = evaluate_zero_shot_combo(group_name, model_name)
        except Exception as exc:
            msg = str(exc)
            error_records.append({'Group': group_name, 'Model': model_name, 'Error': msg})
            print(f'Skipping {group_name} × {model_name}: {msg}')
            continue

        token = f"e1_zero_shot_{safe_name(group_name)}_{safe_name(model_name)}"
        pred_out = RESULTS_DIR / f'{token}_predictions.csv'
        pred_df.to_csv(pred_out)

        plot_pred_vs_actual(
            pred_df['actual_vo2peak'],
            pred_df['predicted_vo2peak'],
            title=f'E1 Zero-shot: {group_name} × {model_name}',
            save_path=IMAGES_DIR / f'{token}_pred_vs_actual.png',
        )
        plot_bland_altman(
            pred_df['actual_vo2peak'],
            pred_df['predicted_vo2peak'],
            title=f'E1 Zero-shot: {group_name} × {model_name} (Bland-Altman)',
            save_path=IMAGES_DIR / f'{token}_bland_altman.png',
        )

        summary_records.append(summary)
        print(f"Done {group_name} × {model_name} | N={summary['NSubjects']} | MAE={summary['MAE']:.3f} | RMSE={summary['RMSE']:.3f}")

e1_summary_df = pd.DataFrame(summary_records)
if not e1_summary_df.empty:
    e1_summary_df = e1_summary_df.sort_values(['Group', 'Model']).reset_index(drop=True)

e1_summary_path = RESULTS_DIR / 'e1_zero_shot_followup_summary.csv'
e1_summary_df.to_csv(e1_summary_path, index=False)
print('Saved E1 summary:', e1_summary_path)

if error_records:
    e1_errors_df = pd.DataFrame(error_records).sort_values(['Group', 'Model']).reset_index(drop=True)
    e1_errors_path = RESULTS_DIR / 'e1_zero_shot_followup_errors.csv'
    e1_errors_df.to_csv(e1_errors_path, index=False)
    print('Saved E1 errors:', e1_errors_path)
    display(e1_errors_df)

e1_summary_df

In [ ]:
if not e1_summary_df.empty:
    best_row = e1_summary_df.sort_values('MAE').iloc[0]
    print('Best E1 zero-shot setting by MAE:')
    print(best_row[['Group', 'Model', 'NSubjects', 'MAE', 'RMSE', 'R2', 'Calibration_slope', 'Calibration_intercept']])

In [ ]:
e1_summary_path = RESULTS_DIR / 'e1_zero_shot_followup_summary.csv'
e1_summary_df = pd.read_csv(e1_summary_path)

group_display = {
    'all_signals': 'All signals',
    'no_gas_exchange': 'No gas exchange',
    'hemo_plus_nirs': 'Hemodynamics + NIRS',
    'nirs': 'NIRS only',
}
group_order = ['all_signals', 'no_gas_exchange', 'hemo_plus_nirs', 'nirs']
model_order = ['Ridge', 'ElasticNet', 'PLS']
group_rank = {name: idx for idx, name in enumerate(group_order)}
model_rank = {name: idx for idx, name in enumerate(model_order)}

report_df = e1_summary_df.copy()
report_df['GroupLabel'] = report_df['Group'].map(group_display).fillna(report_df['Group'])
report_df['GroupOrder'] = report_df['Group'].map(group_rank).fillna(999)
report_df['ModelOrder'] = report_df['Model'].map(model_rank).fillna(999)
report_df = report_df.sort_values(['GroupOrder', 'ModelOrder']).reset_index(drop=True)

table_full = report_df[[
    'GroupLabel', 'Model', 'NSubjects', 'MAE', 'RMSE', 'R2',
    'Pearson_r', 'Pearson_p', 'Spearman_rho',
    'Calibration_slope', 'Calibration_intercept', 'MeanResidual'
]].rename(columns={
    'GroupLabel': 'Group',
    'NSubjects': 'N',
    'Pearson_r': 'Pearson r',
    'Pearson_p': 'Pearson p',
    'Spearman_rho': 'Spearman rho',
    'Calibration_slope': 'Calib. slope',
    'Calibration_intercept': 'Calib. intercept',
    'MeanResidual': 'Mean residual',
})

table_full_csv_path = RESULTS_DIR / 'e1_results_table_full.csv'
table_full_tex_path = RESULTS_DIR / 'e1_results_table_full.tex'
table_full.to_csv(table_full_csv_path, index=False)

table_full_tex = table_full.copy()
for col in ['MAE', 'RMSE', 'R2', 'Pearson r', 'Spearman rho', 'Calib. slope', 'Calib. intercept', 'Mean residual']:
    table_full_tex[col] = table_full_tex[col].map(lambda x: round(float(x), 3))
table_full_tex['Pearson p'] = table_full['Pearson p'].map(lambda x: '<0.001' if float(x) < 0.001 else f"{float(x):.3f}")
table_full_tex.to_latex(
    table_full_tex_path,
    index=False,
    escape=False,
    caption='E1 zero-shot transfer performance on follow-up data across all model-feature group combinations.',
    label='tab:e1_full',
)

best_by_group = report_df.sort_values('MAE').groupby('Group', as_index=False).first()
best_table = best_by_group[[
    'GroupLabel', 'Model', 'NSubjects', 'MAE', 'RMSE', 'R2',
    'Pearson_r', 'Pearson_p', 'Calibration_slope', 'Calibration_intercept', 'MeanResidual'
]].rename(columns={
    'GroupLabel': 'Group',
    'NSubjects': 'N',
    'Pearson_r': 'Pearson r',
    'Pearson_p': 'Pearson p',
    'Calibration_slope': 'Calib. slope',
    'Calibration_intercept': 'Calib. intercept',
    'MeanResidual': 'Mean residual',
})

best_table_csv_path = RESULTS_DIR / 'e1_results_table_best_by_group.csv'
best_table_tex_path = RESULTS_DIR / 'e1_results_table_best_by_group.tex'
best_table.to_csv(best_table_csv_path, index=False)

best_table_tex = best_table.copy()
for col in ['MAE', 'RMSE', 'R2', 'Pearson r', 'Calib. slope', 'Calib. intercept', 'Mean residual']:
    best_table_tex[col] = best_table_tex[col].map(lambda x: round(float(x), 3))
best_table_tex['Pearson p'] = best_table['Pearson p'].map(lambda x: '<0.001' if float(x) < 0.001 else f"{float(x):.3f}")
best_table_tex.to_latex(
    best_table_tex_path,
    index=False,
    escape=False,
    caption='Best-performing model within each feature group for E1 zero-shot transfer.',
    label='tab:e1_best_by_group',
)

mae_matrix = report_df.pivot(index='Group', columns='Model', values='MAE').reindex(group_order)[model_order]
mae_labels = [group_display[g] for g in mae_matrix.index]
mae_values = mae_matrix.to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(7.2, 3.4))
im = ax.imshow(mae_values, cmap='viridis_r', aspect='auto')
ax.set_xticks(np.arange(len(model_order)), labels=model_order)
ax.set_yticks(np.arange(len(mae_labels)), labels=mae_labels)
threshold = np.nanmedian(mae_values)
for i in range(mae_values.shape[0]):
    for j in range(mae_values.shape[1]):
        color = 'white' if mae_values[i, j] > threshold else 'black'
        ax.text(j, i, f'{mae_values[i, j]:.3f}', ha='center', va='center', color=color, fontsize=9)
colorbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
colorbar.set_label('MAE (L/min)')
ax.set_xlabel('Model')
ax.set_ylabel('Feature group')
ax.set_title('E1 zero-shot performance heatmap (MAE)')
fig.tight_layout()
mae_heatmap_path = IMAGES_DIR / 'e1_zero_shot_mae_heatmap.png'
fig.savefig(mae_heatmap_path, dpi=300, bbox_inches='tight')
plt.close(fig)

display(table_full_tex)
display(best_table_tex)
print('Saved:', table_full_csv_path)
print('Saved:', table_full_tex_path)
print('Saved:', best_table_csv_path)
print('Saved:', best_table_tex_path)
print('Saved:', mae_heatmap_path)

In [ ]:
def pretty_group_name(group_code: str) -> str:
    return group_display.get(group_code, group_code.replace('_', ' '))

def format_p_value(p_value: float) -> str:
    return '<0.001' if float(p_value) < 0.001 else f"{float(p_value):.3f}"

e1_sorted = e1_summary_df.sort_values('MAE').reset_index(drop=True)
best_row = e1_sorted.iloc[0]
worst_row = e1_sorted.iloc[-1]
top_three = e1_sorted.head(3)

n_subjects = int(e1_summary_df['NSubjects'].iloc[0])
n_combinations = int(len(e1_summary_df))
best_label = f"{pretty_group_name(best_row['Group'])} + {best_row['Model']}"
worst_label = f"{pretty_group_name(worst_row['Group'])} + {worst_row['Model']}"

top_three_text = '; '.join(
    f"{pretty_group_name(row['Group'])} + {row['Model']} (MAE={row['MAE']:.3f})"
    for _, row in top_three.iterrows()
)

group_mae_mean = e1_summary_df.groupby('Group')['MAE'].mean()
group_r2_mean = e1_summary_df.groupby('Group')['R2'].mean()
group_mae_text = '; '.join(
    f"{pretty_group_name(group_code)}: {group_mae_mean[group_code]:.3f}"
    for group_code in group_order
)
group_r2_text = '; '.join(
    f"{pretty_group_name(group_code)}: {group_r2_mean[group_code]:.3f}"
    for group_code in group_order
)

paragraph_1 = (
    rf"Zero-shot transfer was evaluated in $n={n_subjects}$ follow-up participants across all "
    rf"$4\times 3={n_combinations}$ feature-group/model combinations (Table~\ref{{tab:e1_full}}). "
    rf"Across combinations, absolute error varied markedly, with MAE ranging from "
    rf"{best_row['MAE']:.3f} to {worst_row['MAE']:.3f}~L$\cdot$min$^{{-1}}$. "
    rf"The strongest settings were {top_three_text}, while the weakest setting was "
    rf"{worst_label} (MAE={worst_row['MAE']:.3f}~L$\cdot$min$^{{-1}}$)."
)

paragraph_2 = (
    rf"The best overall model was \textbf{{{best_label}}}, with MAE={best_row['MAE']:.3f}~L$\cdot$min$^{{-1}}$, "
    rf"RMSE={best_row['RMSE']:.3f}~L$\cdot$min$^{{-1}}$, $R^2={best_row['R2']:.3f}$, "
    rf"Pearson $r={best_row['Pearson_r']:.3f}$ ($p={format_p_value(best_row['Pearson_p'])}$), "
    rf"calibration slope={best_row['Calibration_slope']:.3f}, calibration intercept="
    rf"{best_row['Calibration_intercept']:.3f}, and mean residual={best_row['MeanResidual']:.3f}~L$\cdot$min$^{{-1}}$. "
    rf"The negative mean residual indicates a mild systematic underestimation of post-training VO$_2$peak."
)

paragraph_3 = (
    rf"At the feature-group level, mean MAE values were: {group_mae_text}~L$\cdot$min$^{{-1}}$. "
    rf"Mean $R^2$ values by group were: {group_r2_text}. "
    rf"These results indicate that retaining the full multimodal signal set yielded the most accurate and "
    rf"stable transport performance, whereas NIRS-only models generalized poorly (all NIRS-only models had negative $R^2$)."
)

paragraph_4 = (
    r"Table~\ref{tab:e1_best_by_group} summarizes the best model within each feature group. "
    r"Figure~\ref{fig:e1_panels} provides a compact visual summary: (i) MAE heatmap across all combinations, "
    r"(ii) predicted-vs-observed agreement for the best model, and (iii) Bland--Altman residual analysis for the same model."
)

section_text = (
    r"\subsection{E1: Zero-shot transfer to the follow-up cohort}"
    + "\n\n" + paragraph_1
    + "\n\n" + paragraph_2
    + "\n\n" + paragraph_3
    + "\n\n" + paragraph_4
)

figures_text = r"""\begin{figure*}[t]
\centering
\includegraphics[width=0.32\linewidth]{images/e1_zero_shot_mae_heatmap.png}\hfill
\includegraphics[width=0.32\linewidth]{images/e1_zero_shot_all_signals_Ridge_pred_vs_actual.png}\hfill
\includegraphics[width=0.32\linewidth]{images/e1_zero_shot_all_signals_Ridge_bland_altman.png}
\caption{E1 zero-shot transfer performance. Left: MAE heatmap across feature groups and models. Middle: predicted vs observed VO$_2$peak for the best-performing setting (All signals + Ridge). Right: Bland--Altman analysis for the same setting.}
\label{fig:e1_panels}
\end{figure*}
"""

latex_bundle = (
    section_text + "\n\n"
    + "% Auto-generated tables\n"
    + "\\input{results/e1_results_table_full.tex}\n"
    + "\\input{results/e1_results_table_best_by_group.tex}\n\n"
    + figures_text + "\n"
)

latex_section_path = RESULTS_DIR / 'e1_results_section.tex'
latex_bundle_path = RESULTS_DIR / 'e1_results_with_tables_and_figure.tex'
latex_section_path.write_text(section_text + '\n', encoding='utf-8')
latex_bundle_path.write_text(latex_bundle, encoding='utf-8')

print(section_text)
print()
print('Saved:', latex_section_path)
print('Saved:', latex_bundle_path)